In [22]:
%run CertificateClassGroupLean2.0.ipynb
load('IrreducibilityLeanProofWriter.sage')

In [25]:
#Computes a primitive element for the field O/I. 

def RandomPrimitiveElementQuot(K,ideal_gens,p):
    O = K.ring_of_integers()
    I = O.ideal(ideal_gens)
    L.<x> = PolynomialRing(GF(p))
    flag = 0 
    cont = 0 
    while flag == 0 and cont < 200:
        g = O.random_element()
        if g != 0 :
            cont = cont + 1
            poly = g.minpoly()
            F = factor(L(g.minpoly()))
            for f in F:
                if p ** f[0].degree() == I.norm() and ((ZZ[X](f[0])(g)) in I):
                    flag = 1
                    return g, f[0]

In [26]:
#Writes a proof of primality for the ideal given by its generators (as an O-module). The name of the ideal 
# is index by the prime above it and the label. So ideal_namepNlabel. 

def PrimalityCertificate (K, B, ideal_gens, p, ideal_name, label, name_prime_proof):
    O = K.ring_of_integers()
    I = O.ideal(ideal_gens)
    N = I.norm()
    ideal_name_full = ideal_name + str(p) + 'N' + str(label)
    gensc = [elems_to_basis([x], B).list() for x in ideal_gens]
    print(f'def {ideal_name_full} : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (' + ExList(str(gensc)) + ' i)))')
    print('')
    span_label = 'SI'+str(p)+'N'+str(label)
    w = IdealEqSpanCertificateLean(K, ideal_gens, B, span_label, ideal_name_full)
    ik, jk = FindNzEntry(w) 
    print('')
    print(f'lemma N{ideal_name_full} : Nat.card (O ⧸ {ideal_name_full}) = {N} := ')
    print(f" ideal_norm_eq_prod' B _ _ (by decide) {ik} {jk} (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ {span_label})")            
    print('')
    
    if N == p: 
        print(f'lemma isPrime{ideal_name_full} : Ideal.IsPrime {ideal_name_full} := prime_ideal_of_norm_prime {name_prime_proof} _ N{ideal_name_full}')
    else: 
        g, f = RandomPrimitiveElementQuot(K,ideal_gens,p)
        a = list((elems_to_basis([g],B)).transpose()[0])
        c = list((elems_to_basis([ZZ[X](f)(g)],B)).transpose()[0])
        IdealMemZLean(K, B,ideal_gens, c, 'MemC'+ideal_name_full, ideal_name_full, span_label)
        print('')
        CertificateIrrModpFFP(GF(p)[X](f),p,label,sys.stdout)
        print(f'''
def P{ideal_name_full} : PrimeIdeal O {p} where 
  r := {len(B)}
  n := {f.degree()}
  hpos := by decide
  TT := timesTableO
  T := Table
  heq := timesTableT_eq_Table
  I := {ideal_name_full}
  hcard := N{ideal_name_full}
  P := {list(f)}
  f := ofList {list(f)}
  hfeq := by decide
  hirr := irreducible_ofList_ofCertificateIrreducibleZModOfList' P{p}P{label}
  hneq := by decide
  hlen := by decide
  c := !{c}
  a := !{a}
  z := ![1,0,0,0,0]
  hBz := B_one_repr
  hpol := by decide
  hcmem := mem_of_certificate O ℤ _ _ _ _ MemC{ideal_name_full}
  hpmem := by 
    show  _ ∈ {ideal_name_full}.carrier
    rw [ideal_eq_of_IdealEqSpanCertificate O ℤ {span_label}]
    apply Submodule.subset_span
    use 0 ; dsimp ; congr ; norm_num
        
        ''')
        print(f'lemma isPrime{ideal_name_full} : Ideal.IsPrime {ideal_name_full} := PrimeIdeal_isPrime P{ideal_name_full}')
        
        

    
   
   # g, f = RandomPrimitiveElementQuot(K,ideal_gens,p)
    

In [27]:
#Returns a list of truples describing the factorization of (p), the first entry are the generators for the ideal, the second is its multiplicity
# and the third equals 1 if it's principal and 0 otherwise. 
def PrimesBelowGens(K,p):
    O = K.ring_of_integers() 
    F = factor(O.ideal(p))
    ideal_gens = [[[], F[i][1], 0] for i in range(len(F))]
    for i in range(len(F)):
        ideal_gens[i][0] = list(F[i][0].gens_reduced())
        if len(ideal_gens[i][0]) == 1:
            ideal_gens[i][2] = 1 
    return ideal_gens
    

In [118]:
#Primes generator 

#We generate all primes involved in the factorization of an interval of primes [M1, M2]. 
# list_exclude is the list containing the primes that are already proven to be prime in the system. 

def PrimesBelowBound(K,B, M1, M2, list_exclude):
    set_random_seed(10)
    contT = 0 
    for p in primes(M1,M2 + 1):
        F = PrimesBelowGens(K,p)
        l = len(F)
        contT = contT + l 
        if p not in list_exclude :
            print(f'instance hp{p} : Fact (Nat.Prime {p}) := {{out := by norm_num}}')
        
        for i in range(l):
            if l > 1 : 
                PrimalityCertificate(K, B, F[i][0], p, 'I',i,'hp' + str(p) + '.out')
            else: 
                PrimalityCertificate(K, B, F[i][0], p, 'I','','hp' + str(p) + '.out')
    
        ideal_gens = [F[i][0] for i in range(l)]
        exp = [F[i][1] for i in range(l)]

        IteratedMulLean(K, B, ideal_gens, exp, 'rfl', 'rfl', 'I' + str(p) + 'N')
        
        ideal_names_list = []
        
        for i in range(l):
            if l > 1 :
                ideal_names_list = ideal_names_list + F[i][1] * ['I' + str(p) + 'N' + str(i)]
            else :
                ideal_names_list = ideal_names_list + F[i][1] * ['I' + str(p) + 'N']
        
        name_ideals_rep = "[" + ", ".join(ideal_names_list) + "]"
        
        
        print(f'''def PBC{p} : PrimesBelowPCertificate {p} !{name_ideals_rep} where 
  Ip := by 
    intro i 
    fin_cases i ''')
        for i in range(len(ideal_names_list)): 
            print('    exact isPrime' + ideal_names_list[i])
        
        if len(ideal_names_list) >= 2 : 
            print(f'''  hPprod := by 
    simp only [Fin.prod_univ_succ, Fin.prod_univ_zero, mul_one, ← Ideal.mul_assoc]
    dsimp
    rw [ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ MulI{p}N{len(ideal_names_list)-2}, Set.range_unique]
    dsimp ; congr 
    erw [B_int_repr]
    rfl''')  
        else : 
            print(f'''  hPprod := by 
    simp only [Fin.prod_univ_succ, Fin.prod_univ_zero, mul_one, ← Ideal.mul_assoc]
    dsimp
    unfold I{p}N
    rw [Set.range_unique]
    dsimp ; congr 
    erw [B_int_repr]
    rfl''')  
    return contT
            

In [105]:
#R.<X> = PolynomialRing(ZZ)
#K.<a> = NumberField(X^5 + 10*X^3 - 10*X^2 - 15*X - 18)
#B = [1, a, a^2, 1/2*a^3 - 1/2*a, 1/24*a^4 - 1/8*a^3 - 5/24*a^2 + 5/24*a - 1/4]
#B = [K(x) for x in B]
#PrimesBelowBound(K,B, 108, 156, [2,3,5])

#O = K.ring_of_integers()
#I = O.ideal([3,a])

#Z_basis = I.basis()
#elems_to_basis(Z_basis, B)
#matrix(ZZ, (elems_to_basis(Z_basis, B)))
#M = Matrix(K, 3, n)
#W = (((matrix(ZZ, n, n, (elems_to_basis(Z_basis, B)))

In [104]:
#Relations generator. 

#Given a bound < M, and (minimal) list of generators J for the class group (given in order and in terms of their O-generators), we 
# find the relations between the ideals with norm less than B and the generators of the class group. In case 
# an ideal is principal, no relation is shown (as this corresponds to a trivial one)

#Default J_name is J. 
# name_cert([e0,...,em]) is the the name of a proof of J^e0 * ... * J^em. 
# generators_vec([e0,...,em]) is the generators for J^e0 * ... * J^em. We choose to provide this because we need consistency and 
# the function gens_reduced() seems to be not entirely deterministic. 
def RelationsGenerator(K, B, M, J_ideal_gens, J_name, name_cert, generators_vec): 
    s = len(J_ideal_gens)
    #Collects the exponents in the class group. 
    BM = []
    O = K.ring_of_integers()
    if s > 1 :
        J_name = [J_name + str(i) for i in range(s)]
    else : 
        J_name = [J_name]
    J = [O.ideal(J_ideal_gens[i]) for i in range(s)]
    for p in primes(M): 
        F = PrimesBelowGens(K,p)
        l = len(F)
        BMp = []
        for cont in range(l):
            I = O.ideal(F[cont][0])
            if I.norm() < M : 
                # We find (e0, ..., em) such that [I] = [J0]^e0 * ... * [Jm]^em
                expon = [(Cl(I)).exponents()[i] for i in range(s)]
                BMp.append(expon)
                if expon != [0 for i in range(s)] : 
                    set_random_seed(10)
                    JJ = prod([J[i]**(expon[i]) for i in range(s)])
                    # We find the generator for the principal ideal I/(J^e0 * ... * J^em) 
                    genel = (I/(JJ)).gens_reduced()[0]
                    # We find the denominator of this algebraic number, so that we write it as alpha/d. 
                    d = genel.denominator()
                    for t in prime_divisors(d):
                        while (d % t == 0) and ((d//t) * genel in O):
                            d = d // t
                    x = d * genel

                    alpha = elems_to_basis([x], B).list()

                    # These are generators for alpha * (J^e0 * ... * J^em)

                    #We need consistency in the generators for the ideal, with the iterated multiplication

                    #JJ_gens_reduced = (JJ).gens_reduced()
                    JJ_gens_reduced = generators_vec([expon[i] for i in range(s)])

                    #if expon[0] == 3:
                    #    JJ_gens_reduced = [4, -1/4*a^4 + 1/4*a^3 - 7/4*a^2 + 25/4*a + 11/2]
                            

                    #A = [x * i for i in (JJ).gens_reduced()]
                    A = [x * i for i in JJ_gens_reduced]
                    
                    # These are generators for d * I 
                    C = ([d * i for i in F[cont][0]])
                    CC = F[cont][0]

                    Gen1 = [elems_to_basis(A, B).transpose().rows()[j].list() for j in range(len(A))]
                    Gen2 = [elems_to_basis(C, B).transpose().rows()[j].list() for j in range(len(A))]

                    before_div = [elems_to_basis(CC, B).transpose().rows()[j].list() for j in range(len(A))]
                    
                    #JJ_gens = [elems_to_basis((JJ).gens_reduced(), B).transpose().rows()[j].list() for j in range(len((JJ).gens_reduced()))]
                    JJ_gens = [elems_to_basis(JJ_gens_reduced, B).transpose().rows()[j].list() for j in range(len(JJ_gens_reduced))]

                    # We have d * I = alpha * (J^e0 * ... * J^em), so we find the combinations between their generators 
                    # that certify this inequality. 

                    h = [IdealLift(K, B, C, A[i]) for i in range(len(A))]
                    g = [IdealLift(K, B, A, C[i]) for i in range(len(C))]

                    # And represent this in terms of our integral basis. 
                    gp = [[elems_to_basis(g[i], B).transpose().rows()[j].list() for j in range(len(A))] for i in range(len(g))]
                    hp = [[elems_to_basis(h[i], B).transpose().rows()[j].list() for j in range(len(C))] for i in range(len(h))]

                    if s < 2: 
                        J_name_exp = J_name[0] + f' ^ {expon[0]}'
                    else : 
                        J_name_exp = J_name[0] + f' ^ {expon[0]}'
                        for i in range(1,s):
                            J_name_exp = J_name_exp + '*' + J_name[i] + f'^ {expon[i]}'
                            
                    cont2 = 0 
                    
                    if l == 1:
                        cont2 = ''
                    else: 
                        cont2 = cont 

                    print(f'''
def R{p}RS{cont} : IdealMulPrincipalCertificate O ℤ timesTableO ({J_name_exp}) !{alpha} {ExList(str(JJ_gens))}
  {ExList(str(Gen1))} where
    T := Table
    heq := timesTableT_eq_Table
    hI := by
      simp only [pow_succ, pow_one, pow_zero, one_mul]
      {name_cert(expon)}
    hmul := by decide

def E{p}RS{cont} : IdealEqCertificateO O ℤ timesTableO (Ideal.span {{{d}}} * I{p}N{cont2}) (Ideal.span {{B.equivFun.symm !{alpha}}} * {J_name_exp}) 
      {ExList(str(Gen2))} {ExList(str(Gen1))} where 
    T := Table
    heq := timesTableT_eq_Table
    hieq1 := by convert ideal_mul_span_singleton_coe O ℤ  _ _ _ rfl {d} ; decide ; exact {{out := by decide}}
    hieq2 := by 
      exact ideal_eq_principal_mul_of_IdealMulPrincipalCertificate O ℤ _ _ _ _ _ R{p}RS{cont}
    g := {ExList(str(gp))}
    h := {ExList(str(hp))}
    hle1 := by decide
    hle2 := by decide

lemma R{p}N{cont} : Ideal.span {{{d}}} * I{p}N{cont2} =  Ideal.span {{B.equivFun.symm !{alpha}}} * {J_name_exp} := by 
  refine ideal_eq_of_IdealEqCertificateO _ _ _ _ _ _ _ E{p}RS{cont}
                    ''')
        BM = BM + [BMp]
    #BM.reverse()
    #print(BM)
    BMf = [e for sublist in BM for e in sublist]
    return BMf
                
                    
            


In [327]:
#Names for the proofs of J_1^l_1...J_m^l_m  

def name_cert_example(l) : 
    if l == [1]:
        return 'rfl'
    if l == [l[0]] and l[0] > 1:
        return f'exact ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ MulJ{l[0] - 2}'
    else :
        return 0
    

In [106]:
#Cl = K.class_group('pari')
#B = [K(x) for x in B]
#J_ideal_gens = [Cl.gens()[0].ideal().gens_reduced()]
#BM = RelationsGenerator(K,B,155, J_ideal_gens, 'J', name_cert_example)

In [32]:
#Write a term of type PrimesBelowBoundCertificte

#The ideals are names as follow: The i-th prime ideal above p is called I{p}N{i} in case there are more than one 
# prime ideals above p, otherwise the single ideal is I{p}N 

#The terms PrimesBelowPCertificate p that certify the prime factorization of (p) are called PBC{p}

#The proof of the norm of the i-th ideal above p is called N{p}N{i} (and N{p}N in case there is only a single ideal). 

#primes_below_proof is the proof of the primes below M 

def PrimesBelowBoundCertificteGen(K,B,M,primes_below_proof):
    #The list of lists of norms of ideals above p. 
    N = []
    Np = []
    I_names = []
    N_names = []
    for p in primes(M): 
        Lp = []
        Fp = []
        I_names_p = []
        N_names_p = []
        for i in range(len(PrimesBelowGens(K,p))): 
            Lp = Lp + ((PrimesBelowGens(K,p))[i][1])*[(O.ideal((PrimesBelowGens(K,p))[i][0])).norm()]
            Fp = Fp + ((PrimesBelowGens(K,p))[i][1])* [(PrimesBelowGens(K,p))[i][2]]
            if len(PrimesBelowGens(K,p)) == 1 :
                I_names_p = I_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'I{p}N']
                N_names_p = N_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'NI{p}N']
            else: 
                I_names_p = I_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'I{p}N{i}']
                N_names_p = N_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'NI{p}N{i}']
        N = N + [Lp]
        Np = Np + [Fp]
        I_names = I_names + [I_names_p]
        N_names = N_names + [N_names_p]
                   
    m = len(N)
    #Nrm = [[N[i][j][0] for j in range(len(N[i]))] for i in range(m)]
    g = [len(N[i]) for i in range(m)]
        
    #Collects the names of those ideals with norm below M 
    ideals_below = [[I_names[i][j] for j in range(g[i]) if N[i][j] < M] for i in range(m)]

    #collects the principality flag of those ideals with norm below M
    ideals_below_flag_p = [[Np[i][j] for j in range(g[i]) if N[i][j] < M] for i in range(m)]
    
    ideals_below.reverse()
    ideals_below_flag_p.reverse()

    ideals_below_as_string = "[" + ", ".join("[" + ", ".join(sub) + "]" for sub in ideals_below) + "]"

    ideals_below_flatten = [I for sublist in ideals_below for I in sublist]
    ideals_below_flag_p_flatten = [I for sublist in ideals_below_flag_p for I in sublist]
    
    I_names.reverse()
    
    ideals_names_as_string = ["[" + ", ".join(sub) + "]" for sub in I_names]
    
    N_names.reverse()

    
    g.reverse()
    P = [p for p in primes(M)]
    P.reverse()
    N.reverse()
    
    print(f'''def PB{M} : PrimesBelowBoundCertificate O {M} where
  m := {m}
  g := !{g}
  P := !{P}
  hP := {primes_below_proof}
  I := fun i => by
    cases i
    rename_i i h
    interval_cases i ''')
    for i in range(m): 
        print(f'    · exact !{ideals_names_as_string[i]}')
    print(f'''  hC := fun i => by
    cases i
    rename_i i h
    interval_cases i''')
    for i in range(m):
        print(f'    · exact PBC{P[i]}')
    print(f'''  N := fun i => by
    cases i
    rename_i i h
    interval_cases i''')
    for i in range(m):
        print(f'    · exact !{N[i]}')
    print(f'''  hN := fun i => by
     cases i
     rename_i i h
     interval_cases i ''')
    for i in range(m):
        print('''     · dsimp ; intro j
       fin_cases j''')
        for j in range(g[i]):
            print(f'       exact {N_names[i][j]}')
    print(f'  Il := !{ideals_below_as_string}')
    print('''  hIl := by
      intro i
      cases i
      rename_i i h
      interval_cases i
      all_goals rfl''')
    return ideals_below_flatten, ideals_below_flag_p_flatten
    
    
        


In [107]:
#listI, listP = PrimesBelowBoundCertificteGen(K,B,153,'primes_below_153')

In [117]:
#Write a term of type PrimesBelowBoundCertificte

#The ideals are names as follow: The i-th prime ideal above p is called I{p}N{i} in case there are more than one 
# prime ideals above p, otherwise the single ideal is I{p}N 

#The terms PrimesBelowPCertificate p that certify the prime factorization of (p) are called PBC{p}

#The proof of the norm of the i-th ideal above p is called N{p}N{i} (and N{p}N in case there is only a single ideal). 

#primes_below_proof is the proof of the prime integers below M 

#number_interval is the number of intervals in which you want to split up the certificate. 

def PrimesBelowBoundCertificteGenInterval(K,B,M,number_interval):
    #The list of lists of norms of ideals above p. 
    O = K.ring_of_integers()
    N = []
    Np = []
    I_names = []
    N_names = []
    for p in primes(M): 
        Lp = []
        Fp = []
        I_names_p = []
        N_names_p = []
        for i in range(len(PrimesBelowGens(K,p))): 
            Lp = Lp + ((PrimesBelowGens(K,p))[i][1]) * [(O.ideal((PrimesBelowGens(K,p))[i][0])).norm()]
            Fp = Fp + ((PrimesBelowGens(K,p))[i][1]) * [(PrimesBelowGens(K,p))[i][2]]
            if len(PrimesBelowGens(K,p)) == 1 :
                I_names_p = I_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'I{p}N']
                N_names_p = N_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'NI{p}N']
            else: 
                I_names_p = I_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'I{p}N{i}']
                N_names_p = N_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'NI{p}N{i}']
        N = N + [Lp] #List of lists with norms of ideals of O 
        Np = Np + [Fp] #List of lists with flag of principality for the ideals
        I_names = I_names + [I_names_p] #List of lists with names for the ideals
        N_names = N_names + [N_names_p] #List of lists with names for the proofs of the norms of the ideals. 

    m = len(N)
    m1 = floor(m/number_interval) #length of each interval 

    def split_intervals(l): 
        L = []
        for i in range(number_interval):
            if i < number_interval -1 : 
                L = L + [[l[j] for j in range(i*m1, (i+1)*m1)]]
            else: 
                L = L + [[l[j] for j in range(i*m1, len(l))]]
        return L 
    
    g = [len(N[i]) for i in range(m)]
    G = split_intervals(g)
        
    #Collects the names of those ideals with norm below M 
    ideals_below = [[I_names[i][j] for j in range(g[i]) if N[i][j] < M] for i in range(m)]
    Ideals_below = split_intervals(ideals_below)

    #collects the principality flag of those ideals with norm below M
    ideals_below_flag_p = [[Np[i][j] for j in range(g[i]) if N[i][j] < M] for i in range(m)]
    Ideals_below_flag_p = split_intervals(ideals_below_flag_p)

    P = [p for p in primes(M)]
    PP =  split_intervals(P) 
    PP_aux = [[1]] + split_intervals(P) 

    NN = split_intervals(N) 
    NN_names = split_intervals(N_names) 

    II_names = split_intervals(I_names) 

    ideals_below_flatten = [I for sublist in ideals_below for I in sublist]
    ideals_below_flag_p_flatten = [I for sublist in ideals_below_flag_p for I in sublist]


    e = [] 
    for j in range(number_interval): 
        Il = PP_aux[j][-1]
        Iu = max(PP_aux[j+1][-1], (floor((j + 1)/number_interval))*(M-1))
        print('')
        print(f"""lemma PB{M}I{j}_primes :
  ∀ (p : ℕ), p ∈ Set.range !{[p for p in primes(Iu + 1) if Il < p]} ↔ Nat.Prime p ∧ {Il} < p ∧ p ≤ {Iu} := by
  simp_rw [← List.mem_ofFn']
  convert PrimeSieve_mem_of_primesBelow _ 10000 _ ?_ _ _ primes_below_100_proof
  decide ; omega ; omega""")
        
        ideals_below_as_string = "[" + ", ".join("[" + ", ".join(sub) + "]" for sub in Ideals_below[j]) + "]"
        ideals_names_as_string = ["[" + ", ".join(sub) + "]" for sub in II_names[j]]
        e = e + [PP_aux[j][-1]]
        print('')
        print(f'''def PB{M}I{j} : PrimesBelowBoundCertificateInterval O {PP_aux[j][-1]} {max(PP_aux[j+1][-1], (floor((j + 1)/number_interval))*(M-1))} {M} where
  m := {len(PP[j])}
  g := !{G[j]}
  P := !{PP[j]}
  hP := PB{M}I{j}_primes
  I := fun i => by
    cases i
    rename_i i h
    interval_cases i ''')
        for i in range(len(PP[j])): 
            print(f'    · exact !{ideals_names_as_string[i]}')
        print(f'''  hC := fun i => by
    cases i
    rename_i i h
    interval_cases i''')
        for i in range(len(PP[j])):
            print(f'    · exact PBC{PP[j][i]}')
        print(f'''  N := fun i => by
    cases i
    rename_i i h
    interval_cases i''')
        for i in range(len(PP[j])):
            print(f'    · exact !{NN[j][i]}')
        print(f'''  hN := fun i => by
    cases i
    rename_i i h
    interval_cases i ''')
        for i in range(len(PP[j])):
            print('''    · dsimp ; intro j
      fin_cases j''')
            for k in range(G[j][i]):
                print(f'      exact {NN_names[j][i][k]}')
        print(f'  Il := !{ideals_below_as_string}')
        print('''  hIl := by
      intro i
      cases i
      rename_i i h
      interval_cases i
      all_goals rfl''')

    print('')
    print(f'''def PB{M} : PrimesBelowBoundCertificate O {M} := by
  refine primesBelowBoundCertificate_of_Interval O !{e + [M - 1]} {M - 1} rfl rfl ?_ ?_
  · decide
  · intro i
    match i with ''')
    for i in range(number_interval):
        print(f'    | {i} => exact PB{M}I{i}')
    
    return ideals_below_flatten, ideals_below_flag_p_flatten

In [116]:
#listI, listP = PrimesBelowBoundCertificteGenInterval(K,B,156,'sorry', 3)


def PB156I0 : PrimesBelowBoundCertificateInterval O 1 37 156 where
  m := 12
  g := ![5, 3, 5, 3, 3, 1, 1, 3, 3, 1, 3, 1]
  P := ![2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37]
  hP := sorry
  I := fun i => by
    cases i
    rename_i i h
    interval_cases i 
    · exact ![I2N0, I2N1, I2N1, I2N2, I2N2]
    · exact ![I3N0, I3N1, I3N2]
    · exact ![I5N, I5N, I5N, I5N, I5N]
    · exact ![I7N0, I7N1, I7N2]
    · exact ![I11N0, I11N1, I11N2]
    · exact ![I13N]
    · exact ![I17N]
    · exact ![I19N0, I19N1, I19N2]
    · exact ![I23N0, I23N1, I23N2]
    · exact ![I29N]
    · exact ![I31N0, I31N1, I31N2]
    · exact ![I37N]
  hC := fun i => by
    cases i
    rename_i i h
    interval_cases i
    · exact PBC2
    · exact PBC3
    · exact PBC5
    · exact PBC7
    · exact PBC11
    · exact PBC13
    · exact PBC17
    · exact PBC19
    · exact PBC23
    · exact PBC29
    · exact PBC31
    · exact PBC37
  N := fun i => by
    cases i
    rename_i i h
    interval_cases i
    · exact ![2, 2, 2,

In [191]:
#Construct the vector of ideals and the corresponding proofs of the generation of the class group and saturation (prime order case)

#I is the list of (names) of ideals (with multiplicity) below the Minkowski bound. F is the list flagging if the corresponding ideal is principal (1) or 
# not. BM[i][j] is the exponent of the i-th ideal corresponding to the j-th generator of the class group. 
# name_primes_cert corresponds to the name of the certificate for the primes below bound B. 
# B is the bound 
# J_name: name for the generator of the class group. Default is J. 
# q is the (prime) order of the class group 
# ideal_pow_proof is the name of the proof of J^q = (a) 

def ClassGroupPrimeOrderProof(I,F, BM, name_primes_cert,B, J_name, J_not_principal_proof,q,ideal_pow_proof):
    Idedup = []
    Fdedup = []
    for i in range(len(I)):
        if I[i] not in Idedup: 
            Idedup.append(I[i])
            Fdedup.append(F[i])
    n = len(Idedup)
    I_dedup_string = "[" + ", ".join(Idedup) + "]" 
    I_string = "[" + ", ".join(I) + "]" 
    print(f'''def g : Fin {n} → Ideal O := !{I_dedup_string}''')
    print('''''')
    print(f'''lemma primesGenerationBelowAux {{x}} : x ∈ (List.ofFn {name_primes_cert}.Il).flatten ↔ x ∈ List.ofFn g := by
  have h1: (List.ofFn {name_primes_cert}.Il).flatten = {I_string} := by rfl
  have h2: List.ofFn g = {I_dedup_string} := by rfl
  rw [h1, h2]
  simp only [List.mem_cons, List.not_mem_nil, or_false, or_self, or_self_left] ''')
    print('')
    print(f'''lemma primesGenerationBelow : {{I : Ideal O | 0 < I.absNorm ∧ I.IsPrime ∧ I.absNorm < {B}}} ⊆ Set.range g := by
  refine le_primes_below_bound_of_PrimesBelowBoundCertificate' (Subalgebra.equivOfEq _ _ O_integral_closure).toRingEquiv g {name_primes_cert} ?_
  intro x hx
  exact primesGenerationBelowAux.1 hx ''')
    BMs = []
    for i in range(len(BM)):
        BMs.append(",".join([str(i) for i in BM[i]]))
    BMstring = "[" + ";".join(BMs) + "]"
    print('')
    print(f"def BM : Matrix (Fin {n}) (Fin {len(BM[0])}) ℕ := !!{BMstring} ")
    print('')
    print(f'''def g' : Fin {n} → nonZeroDivisors (Ideal O) := by
  intro i
  have : g i ∈ (List.ofFn {name_primes_cert}.Il).flatten := by 
    simp only [primesGenerationBelowAux, List.mem_ofFn, exists_apply_eq_apply]
  exact Ideal.toNonZeroDivisorOfNeZero (g i) 
    (fun hc => ((zero_ne_mem_of_PrimesBelowCertificate _ {name_primes_cert}) (hc ▸ this) ))

def x : Fin {len(BM[0])} → Ideal O := ![{J_name}]

def x' :  Fin {len(BM[0])} → nonZeroDivisors (Ideal O) := by 
  refine fun i => Ideal.toNonZeroDivisorOfNeZero (x i) (fun hc => (J_not_principal ⟨0, ?_⟩ ))
  ext 
  simp only [x, Fin.fin_one_eq_zero i, Fin.isValue, cons_val_fin_one, ne_eq] at hc
  simp only [hc, Ideal.mem_span_singleton, Ideal.mem_bot, zero_dvd_iff]

lemma g'_apply : ∀ (i : Fin {n}), ↑(g' i) = g i := by
  intro i
  rfl

lemma x'_apply : ∀ (i : Fin {len(BM[0])}), ↑(x' i) = x i := by
  intro i
  rfl

theorem class_group_generator : Subgroup.closure (Set.range (fun i => ClassGroup.mk0 (x' i))) = ⊤ := by
    refine subgroup_closure_eq_classGroup'' (r := Fin 1) (Subalgebra.equivOfEq _ _ O_integral_closure).toRingEquiv
      g'_apply x'_apply ?_ primesGenerationBelow BM ?_
    · refine lt_of_le_of_lt K_minowski ?_
      norm_num
    · simp only [Finset.univ_unique, Fin.default_eq_zero, Fin.isValue, Finset.prod_singleton]
      intro i
      fin_cases i ''')
    for i in range(len(Idedup)): 
        if Fdedup[i] != 1:
            print(f'''      · refine Exists.intro _ (Exists.intro ?_ (Exists.intro (?_) (Exists.intro ?_ (by convert R{(Idedup[i])[1:]}))))
        refine Nat.cast_ne_zero.2 (by decide)
        exact (LinearEquiv.map_ne_zero_iff B.equivFun.symm).mpr (by decide)''')
        else: 
            print('''      · exact ideal_mem_principal_class' O ℤ B _ _ (by decide) rfl''')
    print('')
    print(f'''def class_group_O_equiv : ZMod {q} ≃+ Additive (ClassGroup O) := by
  refine equivClassGroupCyclicOfSaturated (I := {J_name}) (I' := x' 0) (rfl) {ideal_pow_proof} ?_ ?_
  convert class_group_generator
  intro p hp hpdvd
  have : p = {q} := ((prime_dvd_prime_iff_eq (q := {q}) (Nat.prime_iff.mp hp)) 
    (Nat.prime_iff.mp (by decide))).1 hpdvd
  simp only [this, Nat.ofNat_pos, Nat.div_self, pow_one]
  exact {J_not_principal_proof}

def class_group_equiv : ZMod {q} ≃+ Additive (ClassGroup (NumberField.RingOfIntegers K)) := by
  refine AddEquiv.trans class_group_O_equiv ?_
  exact MulEquiv.toAdditive (ClassGroup.congr
    ((Subalgebra.equivOfEq _ _ O_integral_closure).toRingEquiv))

theorem class_number_K_eq_{q} : NumberField.classNumber K = {q} := by
  unfold NumberField.classNumber
  rw [Fintype.card_eq_nat_card, ← Nat.card_congr (Additive.toMul),
    Nat.card_congr (class_group_equiv).symm.toEquiv]
  exact Nat.card_eq_fintype_card
    ''')
    

In [109]:
#ClassGroupPrimeOrderProof(listI, listP,BM,'PB153',156, 'J', 'J_not_principal',5,'J5')

In [ ]:
# set_random_seed(10)

# #Compute class group and let J be the generator. 

# Cl = K.class_group('pari')
# n = K.degree()
# a = K.gen()
# B = [K(x) for x in B]
# ideal_gens = Cl.gens()[0].ideal().gens_reduced()
# J = O.ideal(ideal_gens)

# p = 7
# F = PrimesBelowGens(K,p)
# l = len(F)

# #The ideals above p with norm less than B 

# ideal_gens = [O.ideal(F[i][0]) for i in range(l) if O.ideal(F[i][0]).norm() < 63]
# print(ideal_gens)


# cont = 0
# expon = (Cl(ideal_gens[cont])).exponents()[0]
# genel = (ideal_gens[cont]/(J**expon)).gens_reduced()[0]
# d = genel.denominator()

# for p in prime_divisors(d):
#         while (d % p == 0) and ((d//p) * genel in O):
#             d = d // p

# #d = 2
# print('holaaaa',d)
# x = d * genel
# #print([x * i for i in (J ** 2).reduced_gens()])
# A = [x * i for i in (J ** expon).gens_reduced()]
# C = ([d * i for i in (ideal_gens[cont]).gens()])
# CC = ideal_gens[cont].gens()

# Gen1 = [elems_to_basis(A, B).transpose().rows()[j].list() for j in range(len(A))]
# Gen2 = [elems_to_basis(C, B).transpose().rows()[j].list() for j in range(len(A))]
# print([elems_to_basis(A, B).transpose().rows()[j].list() for j in range(len(A))])
# print([elems_to_basis(C, B).transpose().rows()[j].list() for j in range(len(A))])

# before_div = [elems_to_basis(CC, B).transpose().rows()[j].list() for j in range(len(A))]
# print('before div =', [elems_to_basis(CC, B).transpose().rows()[j].list() for j in range(len(A))])
# #print(elems_to_basis(C, B).transpose())
# h = [IdealLift(K, B, C, A[i]) for i in range(len(A))]
# g = [IdealLift(K, B, A, C[i]) for i in range(len(C))]

# alpha = elems_to_basis([x], B).list()

# print('alpha = ', alpha, 'denom =',d, 'exponent', expon)


# gp = [[elems_to_basis(g[i], B).transpose().rows()[j].list() for j in range(len(A))] for i in range(len(g))]
# hp = [[elems_to_basis(h[i], B).transpose().rows()[j].list() for j in range(len(C))] for i in range(len(h))]

# print(ExList(str(gp)))
# print(ExList(str(hp)))

# numero = cont 

# print(f'''
# def R{p}RS{numero} : IdealMulPrincipalCertificate O ℤ timesTableO (J ^ {expon}) !{alpha} ![![2, 0, 0, 0, 0], ![1, 0, 1, 0, 0]]
#   {ExList(str(Gen1))} where
# T := Table
# heq := timesTableT_eq_Table
# hI := by
#   simp only [pow_succ, pow_one, pow_zero, one_mul]
#   exact ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ MulJ0
# hmul := by decide

# def E{p}RS{numero} : IdealEqCertificateO O ℤ timesTableO (Ideal.span {d} * I{p}N{numero}) (Ideal.span B.equivFun.symm !{alpha} * J ^ {expon}) 
#   {ExList(str(Gen2))} {ExList(str(Gen1))} where 
# T := Table
# heq := timesTableT_eq_Table
# hieq1 := by 
#   unfold I{p}N{numero}
#   rw [Ideal.span_mul_span] ; congr
#   simp only [Set.mem_singleton_iff, Set.mem_range, Set.iUnion_exists,
#     Set.iUnion_iUnion_eq', Set.iUnion_singleton_eq_range, Set.iUnion_iUnion_eq_left, (show (2 : O) = ↑(2 : ℤ) by rfl), 
#     ← zsmul_eq_mul _ 2, ← LinearEquiv.map_smul, B]
#   have aux : (2 : ℤ) • {ExList(str(before_div))} = {ExList(str(Gen2))} := by decide
#   refine congr_arg (fun x => Set.range x) ?_
#   ext i ; congr  ; exact congr_fun aux i 
# hieq2 := by 
#   exact ideal_eq_principal_mul_of_IdealMulPrincipalCertificate O ℤ _ _ _ _ _ R{p}RS{numero}
# g := {ExList(str(gp))}
# h := {ExList(str(hp))}
# hle1 := by decide
# hle2 := by decide

# lemma R{p}N{numero} : Ideal.span {d} * I{p}N{numero} =  Ideal.span B.equivFun.symm !{alpha} * J ^ {expon} := by 
#   refine ideal_eq_of_IdealEqCertificateO _ _ _ _ _ _ _ E{p}RS{numero}




# ''')


In [110]:
#def IteratedMulLean (K, B, ideal_gens, ideal_pows, proof1_name, proof2_name, ideal_name):

#IteratedMulLean(K, B, [[3,a], [3, -5/24*a^4 + 1/8*a^3 - 47/24*a^2 + 59/24*a + 13/4],[3,a + 1]], [1,1,1], 'rfl', 'rfl', 'I3N')

In [111]:
# list_ideal_gens = [3,a]
# ideal_name = 'I'

# gensc = [elems_to_basis([x], B).list() for x in list_ideal_gens]
# print(f'def {ideal_name} : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (' + ExList(str(gensc)) + ' i)))')

# w = IdealEqSpanCertificateLean(K, list_ideal_gens, B, 'A', ideal_name)
# ik, jk = FindNzEntry(w) 
# print('')
# print(f'lemma N : Nat.card (O ⧸ {ideal_name}{i}) = {O.ideal(list_ideal_gens).norm()} := ')
# print(f" ideal_norm_eq_prod' B _ _ (by decide) {ik} {jk} (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ A)")            
# print('')

In [143]:
#IdealMemZLean(K, B,list_ideal_gens, [5583, -3658, 3709, 4702, 22416], 'MemC', 'I', 'A')

def MemC : IdealMemCertificate O ℤ B I
  ![![3, 0, 0, 0, 0], ![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 0, 0, 1, 0], ![0, 0, 0, 0, 3]] ![5583, -3658, 3709, 4702, 22416] where
 hieq := ideal_eq_of_IdealEqSpanCertificate _ _ A
 g := ![1861, -3658, 3709, 4702, 7472]
 hmem := by decide


In [112]:
#import sys
#CertificateIrrModpFFP(GF(3)[X](f),3,0,sys.stdout)

In [123]:
[p for p in primes(5 + 1) if 2 < p ]

[3, 5]